# SAE Sweep Analysis

This notebook compares SAE runs across the hyperparameter sweep to help choose
the best `k` (sparsity) and `n_latent` (dictionary size) per layer.

**Key metrics for evaluating an SAE:**

| Metric | What it tells you | Goal |
|---|---|---|
| **Variance explained** | How much of the original signal the SAE reconstructs | Higher is better (≥0.90 is good) |
| **MSE** | Raw reconstruction error per dimension | Lower is better |
| **Dead features (%)** | Fraction of latent features that never activate | Lower is better (<10% ideal) |
| **Feature frequency distribution** | How evenly features are used | Even spread preferred over a few dominant features |

**Trade-offs to consider:**
- **k** (sparsity): Lower k → more interpretable features but worse reconstruction.
  Higher k → better reconstruction but features may be less monosemantic.
- **n_latent** (dictionary size): Larger → more capacity to capture distinct features
  but more dead features and higher compute cost.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path.cwd().parent))
from sae.data import EmbeddingDataset, LAYER_NAMES
from sae.eval import load_sae, compute_metrics

In [ ]:
# --- Configuration: update these paths ---
SWEEP_DIR = Path("/ocean/projects/atm170004p/jshen6/cae_sae/")
DATA_DIR = Path("/ocean/projects/atm170004p/lxu5/ConvAE/EVAL_DATA/SNAPSHOTS/CAEwCP_TrP_A_TrS_10000_KS_5_LN_5_LD_256_BS_2_LR_0.001_WD_1e-06_DP_0.0_EN_1500_SD_42/ep_best_pi_0/TeP_A_10000_TD_0.2_SAE/")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LAYERS = ["IN", "E1", "E2", "E3", "E4", "E5", "D1", "D2", "D3", "D4", "D5", "OUT"]
K_VALUES = [16, 32, 64, 128]
N_LATENT_VALUES = [2048, 4096, 8192]

print(f"Sweep dir: {SWEEP_DIR}")
print(f"Device: {DEVICE}")

## 1. Collect metrics from all runs

Load each checkpoint, run evaluation, and gather results into a DataFrame.

In [ ]:
# Cache layer data to avoid reloading for every run on the same layer
layer_data_cache = {}

def get_layer_data(layer_name):
    if layer_name not in layer_data_cache:
        ds = EmbeddingDataset(DATA_DIR / f"{layer_name}.pt")
        layer_data_cache[layer_name] = ds.data
    return layer_data_cache[layer_name]


rows = []
missing = []

for layer in LAYERS:
    data = get_layer_data(layer)
    for k in K_VALUES:
        for n_latent in N_LATENT_VALUES:
            run_name = f"{layer}_k{k}_n{n_latent}"
            ckpt_path = SWEEP_DIR / run_name / "best.pt"

            if not ckpt_path.exists():
                missing.append(run_name)
                continue

            model, ckpt = load_sae(ckpt_path, device=DEVICE)
            metrics = compute_metrics(model, data, device=DEVICE)

            rows.append({
                "layer": layer,
                "k": k,
                "n_latent": n_latent,
                "mse": metrics["mse"],
                "variance_explained": metrics["variance_explained"],
                "n_dead": metrics["n_dead"],
                "dead_pct": metrics["n_dead"] / n_latent * 100,
                "feature_freq": metrics["feature_freq"],
            })
            print(f"  {run_name}: VE={metrics['variance_explained']:.4f}, "
                  f"dead={metrics['n_dead']}/{n_latent} ({metrics['n_dead']/n_latent*100:.1f}%)")

    # Free memory after each layer
    del layer_data_cache[layer]
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

if missing:
    print(f"\n{len(missing)} runs missing (not yet completed or failed):")
    for m in missing[:10]:
        print(f"  {m}")
    if len(missing) > 10:
        print(f"  ... and {len(missing) - 10} more")

# Separate feature_freq from the main DataFrame
feature_freqs = {(r["layer"], r["k"], r["n_latent"]): r.pop("feature_freq") for r in rows}
df = pd.DataFrame(rows)
print(f"\nCollected {len(df)} runs.")
df.head()

## 2. Variance Explained: the primary quality metric

Variance explained tells you what fraction of the original signal your SAE preserves.
A good SAE should achieve ≥0.90 (90%). Below ~0.80 means the SAE is losing important information.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12), sharey=True)
axes = axes.flatten()

for i, layer in enumerate(LAYERS):
    ax = axes[i]
    sub = df[df["layer"] == layer]
    if sub.empty:
        ax.set_title(f"{layer} (no data)")
        continue

    for n_latent in N_LATENT_VALUES:
        mask = sub["n_latent"] == n_latent
        ax.plot(sub.loc[mask, "k"], sub.loc[mask, "variance_explained"],
                "o-", label=f"n={n_latent}")

    ax.set_title(layer, fontweight="bold")
    ax.set_xlabel("k")
    ax.set_xticks(K_VALUES)
    ax.axhline(0.90, color="gray", ls="--", alpha=0.5, label="90% threshold")
    if i == 0:
        ax.legend(fontsize=8)

axes[0].set_ylabel("Variance Explained")
axes[4].set_ylabel("Variance Explained")
axes[8].set_ylabel("Variance Explained")
fig.suptitle("Variance Explained vs. k (grouped by n_latent)", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 3. Dead Features: wasted capacity

Dead features are latent dimensions that never activate. They represent wasted model capacity.
- **>30% dead**: dictionary is too large for this k, or training didn't converge
- **10-30% dead**: acceptable, but could use a smaller dictionary
- **<10% dead**: good utilization

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12), sharey=True)
axes = axes.flatten()

for i, layer in enumerate(LAYERS):
    ax = axes[i]
    sub = df[df["layer"] == layer]
    if sub.empty:
        ax.set_title(f"{layer} (no data)")
        continue

    for n_latent in N_LATENT_VALUES:
        mask = sub["n_latent"] == n_latent
        ax.plot(sub.loc[mask, "k"], sub.loc[mask, "dead_pct"],
                "o-", label=f"n={n_latent}")

    ax.set_title(layer, fontweight="bold")
    ax.set_xlabel("k")
    ax.set_xticks(K_VALUES)
    ax.axhline(10, color="green", ls="--", alpha=0.5)
    ax.axhline(30, color="red", ls="--", alpha=0.5)
    if i == 0:
        ax.legend(fontsize=8)

axes[0].set_ylabel("Dead Features (%)")
axes[4].set_ylabel("Dead Features (%)")
axes[8].set_ylabel("Dead Features (%)")
fig.suptitle("Dead Feature % vs. k (grouped by n_latent)", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 4. Reconstruction vs. Sparsity Trade-off (Pareto Front)

The core SAE trade-off: sparser representations (lower k) are more interpretable
but reconstruct worse. This plot helps identify the "sweet spot" where you get
good reconstruction without excessive sparsity.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, layer in enumerate(LAYERS):
    ax = axes[i]
    sub = df[df["layer"] == layer]
    if sub.empty:
        ax.set_title(f"{layer} (no data)")
        continue

    for n_latent in N_LATENT_VALUES:
        mask = sub["n_latent"] == n_latent
        s = sub[mask].sort_values("k")
        ax.plot(s["k"], s["variance_explained"], "o-", label=f"n={n_latent}")
        # Annotate dead% next to each point
        for _, row in s.iterrows():
            ax.annotate(f"{row['dead_pct']:.0f}%",
                       (row["k"], row["variance_explained"]),
                       textcoords="offset points", xytext=(5, 5), fontsize=6, alpha=0.7)

    ax.set_title(layer, fontweight="bold")
    ax.set_xlabel("k (sparsity)")
    ax.set_ylabel("Variance Explained")
    ax.set_xticks(K_VALUES)
    if i == 0:
        ax.legend(fontsize=7)

fig.suptitle("Reconstruction vs. Sparsity (annotations = dead%)", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 5. Feature Frequency Distributions

A healthy SAE should have features that fire at a range of frequencies:
- If most features fire very rarely → dictionary may be too large
- If a few features dominate → features may be polysemantic (encoding multiple concepts)
- A roughly log-uniform spread is typical and healthy

In [ ]:
# Pick a representative layer to show feature frequency distributions
sample_layer = "E3"  # Change to whichever layer you care about most

fig, axes = plt.subplots(len(N_LATENT_VALUES), len(K_VALUES),
                          figsize=(16, 3 * len(N_LATENT_VALUES)), sharex=True)

for i, n_latent in enumerate(N_LATENT_VALUES):
    for j, k in enumerate(K_VALUES):
        ax = axes[i, j]
        key = (sample_layer, k, n_latent)
        if key not in feature_freqs:
            ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
            continue

        freq = feature_freqs[key].numpy()
        alive = freq[freq > 0]

        if len(alive) > 0:
            ax.hist(np.log10(alive + 1e-10), bins=50, alpha=0.7, color="steelblue")
        ax.set_title(f"k={k}, n={n_latent}\n({len(alive)}/{len(freq)} alive)", fontsize=9)
        if i == len(N_LATENT_VALUES) - 1:
            ax.set_xlabel("log10(freq)")
        if j == 0:
            ax.set_ylabel("Count")

fig.suptitle(f"Feature Frequency Distributions — Layer {sample_layer}", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 6. Summary Heatmaps

Quick visual comparison across all layers and hyperparameter combinations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for idx, n_latent in enumerate(N_LATENT_VALUES):
    ax = axes[idx]
    sub = df[df["n_latent"] == n_latent]
    if sub.empty:
        continue

    pivot = sub.pivot(index="layer", columns="k", values="variance_explained")
    # Reorder rows to match layer order
    pivot = pivot.reindex([l for l in LAYERS if l in pivot.index])

    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0.5, vmax=1.0)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("k")
    ax.set_title(f"n_latent = {n_latent}")

    # Annotate cells
    for ii in range(len(pivot.index)):
        for jj in range(len(pivot.columns)):
            val = pivot.values[ii, jj]
            if not np.isnan(val):
                ax.text(jj, ii, f"{val:.3f}", ha="center", va="center", fontsize=8,
                       color="black" if val > 0.7 else "white")

fig.colorbar(im, ax=axes, label="Variance Explained", shrink=0.8)
fig.suptitle("Variance Explained — All Layers × k × n_latent", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for idx, n_latent in enumerate(N_LATENT_VALUES):
    ax = axes[idx]
    sub = df[df["n_latent"] == n_latent]
    if sub.empty:
        continue

    pivot = sub.pivot(index="layer", columns="k", values="dead_pct")
    pivot = pivot.reindex([l for l in LAYERS if l in pivot.index])

    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=50)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("k")
    ax.set_title(f"n_latent = {n_latent}")

    for ii in range(len(pivot.index)):
        for jj in range(len(pivot.columns)):
            val = pivot.values[ii, jj]
            if not np.isnan(val):
                ax.text(jj, ii, f"{val:.0f}%", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=axes, label="Dead Features (%)", shrink=0.8)
fig.suptitle("Dead Features % — All Layers × k × n_latent", fontsize=14)
fig.tight_layout()
plt.show()

## 7. Recommended Configurations

Automatically select the best config per layer using a simple scoring rule:
- Variance explained ≥ 0.90 (hard constraint)
- Among those, prefer lower k (more interpretable), then lower dead%

In [ ]:
VE_THRESHOLD = 0.90

recommendations = []
for layer in LAYERS:
    sub = df[(df["layer"] == layer) & (df["variance_explained"] >= VE_THRESHOLD)]
    if sub.empty:
        # Fallback: pick the best VE regardless
        sub = df[df["layer"] == layer]
        if sub.empty:
            continue
        best = sub.loc[sub["variance_explained"].idxmax()]
        recommendations.append({
            "layer": layer,
            "k": int(best["k"]),
            "n_latent": int(best["n_latent"]),
            "VE": best["variance_explained"],
            "dead%": best["dead_pct"],
            "note": "below threshold — best available",
        })
    else:
        # Among configs meeting the VE threshold, prefer lowest k, then lowest dead%
        sub_sorted = sub.sort_values(["k", "dead_pct"])
        best = sub_sorted.iloc[0]
        recommendations.append({
            "layer": layer,
            "k": int(best["k"]),
            "n_latent": int(best["n_latent"]),
            "VE": best["variance_explained"],
            "dead%": best["dead_pct"],
            "note": "",
        })

rec_df = pd.DataFrame(recommendations)
print("Recommended configs per layer:")
print(f"(threshold: VE >= {VE_THRESHOLD}, then prefer lowest k and dead%)\n")
print(rec_df.to_string(index=False))

## Interpretation Guide

**What to look for in the results above:**

1. **Check variance explained heatmap first.** If all configs for a layer are below 0.90,
   the SAE may need more training epochs, a different architecture, or the layer embeddings
   may simply be hard to sparsify.

2. **Look at diminishing returns in k.** If going from k=32 to k=64 barely improves VE,
   then k=32 is likely sufficient — fewer active features means more interpretable features.

3. **Check if larger dictionaries help.** If n_latent=4096 and n_latent=8192 give similar VE
   but 8192 has more dead features, the extra capacity is wasted — use 4096.

4. **Feature frequency distributions matter.** A good SAE has features firing across a
   range of frequencies. If most features are ultra-rare (<0.01 freq), the dictionary is too large.
   If a few features fire on >50% of inputs, they may be encoding the mean rather than
   meaningful structure.

5. **Layer-specific behavior is expected.** Early/late layers (IN, OUT) often behave differently
   from bottleneck layers (E5, D1). The bottleneck may need different hyperparameters.

6. **The recommended configs are a starting point.** If you need maximum interpretability,
   push k lower even at the cost of some VE. If you need faithful reconstruction, allow higher k.